# 12 · 프레임워크 통합 (DLPack · PyTorch 무복사 연동)

> **CuPy 2일 집중 코스 — Day 2 / 단원 9 (실전 예제)**

CuPy 배열을 **복사 없이(zero-copy)** PyTorch 등 다른 GPU 라이브러리와 주고받는 법을 배웁니다.
같은 GPU 메모리를 공유하므로 전처리는 CuPy로, 모델은 PyTorch로 — 전송 없이 연결할 수 있습니다.

### 왜 상호운용을 배우는가
실전 파이프라인은 한 라이브러리로 끝나지 않습니다. 데이터 적재·전처리는 CuPy/cuDF가 강하고,
모델 학습·추론은 PyTorch/TensorFlow가 강합니다. 문제는 두 세계 사이에서 배열을 어떻게 넘기느냐입니다.
가장 순진한 방법 — `torch.tensor(cp.asnumpy(x)).cuda()` — 은 GPU → host → GPU를 왕복하는
**두 번의 PCIe 전송**을 발생시킵니다. `00_intro_env`에서 본 것처럼 PCIe Gen4 x16 대역폭은
편도 약 32 GB/s 수준이므로, 예컨대 400MB(1억 개 float32) 배치라면 왕복에 20~30ms 안팎이 그냥
사라집니다 — 매 미니배치마다 이 비용을 반복하면 GPU 연산 자체보다 전송이 병목이 되는 경우가
흔합니다. **DLPack**과 **CUDA Array Interface** 같은 상호운용 표준은 이 왕복을 없애고, 같은
물리 GPU 메모리를 가리키는 **포인터만 공유**합니다 — 비용은 사실상 메타데이터(포인터·shape·dtype·
strides) 몇십~몇백 바이트를 복사하는 수준이라 데이터 크기와 무관하게 마이크로초 이하입니다.

### 이 노트북의 흐름
개념(왜 상호운용인가 → DLPack 표준) → 실습(CuPy→PyTorch → PyTorch→CuPy → `__cuda_array_interface__`)
→ 함정(소유권·동기화 주의점) → 연습(무복사 파이프라인 구성) → 부록(Numba↔CuPy, 연속성/스트림 심화).
이 노트북에서 배우는 무복사 연결 패턴은 `13_dl_preprocess_capstone`에서 **커스텀 CUDA 커널로
전처리한 데이터를 그대로 PyTorch 학습 루프에 흘려보내는** 종합 실습으로 이어집니다.

## 학습 목표
- **DLPack** 표준과 `__cuda_array_interface__`로 무복사 교환을 이해한다.
- CuPy ↔ PyTorch 텐서를 **포인터 공유**로 변환한다.
- 소유권·동기화 주의점을 안다.

## 목차
1. [왜 상호운용인가](#1) · 2. [DLPack 표준](#2) · 3. [CuPy → PyTorch](#3)
4. [PyTorch → CuPy](#4) · 5. [`__cuda_array_interface__`](#5) · 6. [주의점](#6) · 7. [연습 & 체크포인트](#7)

> `torch`(CUDA 빌드)가 있으면 실습이 동작합니다. 없으면 안내만 출력하고 개념은 그대로 학습합니다.

이 `HAS_TORCH` 방어 패턴(try/except로 감지 후 분기)은 이 코스에서 선택적 의존성을 다룰 때
반복되는 관례입니다(예: `08_numba_basics` 계열의 `numba` 감지). 실습 서버에 `torch`가 없어도
노트북이 예외 없이 끝까지 실행되도록 하기 위함이며, 실제 강의장 환경처럼 GPU·라이브러리 구성이
사람마다 다를 수 있는 상황을 감안한 설계입니다.

In [ ]:
import numpy as np, cupy as cp
from course_utils import print_env, allclose
print_env()
try:
    import torch
    HAS_TORCH = torch.cuda.is_available()
    print('torch CUDA 사용 가능:', HAS_TORCH)
except Exception as e:
    HAS_TORCH = False
    print('torch 미설치 — 개념만 학습합니다. (', e, ')')

<a id="1"></a>
## 1. 왜 상호운용인가

딥러닝 파이프라인은 보통 **전처리(CuPy/cuDF) → 모델(PyTorch/TF)** 로 이어집니다.
- 라이브러리마다 따로 GPU 배열을 복사하면 **전송 낭비**
- 같은 GPU 메모리를 **포인터로 공유**하면 복사 0 → 빠름
- 표준: **DLPack**, **CUDA Array Interface**(`__cuda_array_interface__`)

### 조금 더 구체적으로

NVIDIA RAPIDS 생태계(cuDF·CuPy)와 PyTorch를 함께 쓰는 추천시스템·표 형태 데이터 파이프라인이
대표적인 예입니다. cuDF로 대규모 테이블을 GPU에서 정제하고, CuPy로 피처를 표준화한 뒤,
DLPack으로 PyTorch 텐서에 그대로 얹어 학습합니다. 만약 매 스텝마다 "CuPy → NumPy(`asnumpy`)
→ `torch.tensor` → `.cuda()`" 경로를 탄다면:
1. device→host 복사 (PCIe, ~수~수십 ms급, 크기에 비례)
2. host→device 복사 (다시 PCIe, 같은 비용)
3. 그 사이 CPU에서 파이썬 객체 생성 오버헤드까지 추가

반면 `from_dlpack()` 경로는 **데이터를 전혀 옮기지 않고**, "이 포인터를 내 텐서로 봐라"는
메타데이터만 교환하므로 배열이 1KB든 10GB든 변환 비용이 거의 일정합니다(포인터·shape·dtype·
strides 등 작은 구조체 하나). 이 차이가 이 노트북 전체를 관통하는 핵심 동기입니다 — **"복사냐,
공유냐"** 가 곧 "밀리초 단위 지연이냐, 마이크로초 이하냐"의 차이로 이어집니다.

<a id="2"></a>
## 2. DLPack 표준

**DLPack**은 프레임워크 간 텐서를 무복사로 교환하는 표준입니다. 최신 API는 `from_dlpack()` 하나로 통일됩니다
(객체가 `__dlpack__`를 구현하면 됨). 구버전은 `toDlpack()`/`fromDlpack(capsule)`을 씁니다.

### 조금 더 구체적으로: DLPack이 실제로 주고받는 것

DLPack의 핵심은 `DLManagedTensor`라는 아주 단순한 C 구조체입니다 — 대략 다음 정보만 담습니다.

| 필드 | 의미 |
|------|------|
| `data` | GPU(또는 CPU) 메모리의 **원시 포인터**(정수) |
| `device` | 디바이스 종류(CUDA/CPU/ROCm 등)와 디바이스 번호 |
| `ndim`, `shape`, `strides` | 배열의 차원·모양·보폭(연속성 정보) |
| `dtype` | 원소 타입(float32, int64 등) |
| `byte_offset` | 데이터 시작 오프셋 |
| `deleter` | 이 뷰가 더 이상 필요 없을 때 호출되는 콜백(원본 소유자에게 "다 썼다"고 알림) |

즉 DLPack은 **데이터를 복사하지 않고, 이 구조체 하나(수십~수백 바이트)만** 새 프레임워크에 넘깁니다.
이 구조체는 파이썬의 `PyCapsule`(불투명 포인터를 담는 표준 컨테이너)에 담겨 `__dlpack__()` 메서드로
노출되고, `from_dlpack(x)`는 내부적으로 `x.__dlpack__()`을 호출해 캡슐을 받아 자기 프레임워크의
텐서 객체로 감쌉니다. `__dlpack_device__()`는 어느 디바이스에 데이터가 있는지만 빠르게 알려주는
보조 메서드입니다.

DLPack은 원래 Apache TVM 생태계(DMLC 프로젝트)에서 프레임워크 간 텐서 교환을 위해 만들어진
**ABI(응용 프로그램 바이너리 인터페이스) 수준의 표준**입니다. C 구조체 기반이라 파이썬뿐 아니라
C/C++ 레벨에서도 그대로 통할 수 있다는 것이 CUDA Array Interface(5절)와의 큰 차이입니다.

> **구버전 API 주의**: `toDlpack()`/`fromDlpack(capsule)`은 캡슐을 **한 번만 소비**할 수 있어
> (재사용하면 에러) 사용이 번거로웠습니다. 최신 `from_dlpack()` 프로토콜(파이썬 배열 API 표준,
> Array API 명세의 일부)은 이런 제약 없이 통일된 한 방식으로 어떤 라이브러리 텐서든 주고받을 수
> 있게 정리한 버전입니다. 이 노트북과 이후 실습에서는 최신 API만 사용합니다.

In [ ]:
# CuPy 내부 DLPack 왕복 (포인터 동일성 확인)
x = cp.arange(10, dtype=cp.float32)
y = cp.from_dlpack(x)          # 무복사
print('같은 메모리?', x.data.ptr == y.data.ptr)   # True

<a id="3"></a>
## 3. CuPy → PyTorch (무복사)

`torch.from_dlpack(cupy_array)` 로 CuPy 배열을 PyTorch 텐서로 (복사 없이) 봅니다.

### 조금 더 구체적으로

내부적으로는 CuPy 배열의 `__dlpack__()`이 반환한 캡슐을, PyTorch가 자신의 `torch.Tensor`로
감싸는 것뿐입니다. 새 GPU 메모리를 할당하지 않으므로 결과 텐서의 `data_ptr()`은 원본 CuPy
배열의 `data.ptr`과 **정확히 같은 정수 값**입니다(아래 셀에서 직접 비교). 이는 두 객체가
**같은 물리 메모리를 가리키는 서로 다른 파이썬 뷰**라는 뜻이며, 한쪽에서 in-place 연산(`+=`,
`.add_()` 등)을 하면 다른 쪽에서도 즉시 그 변화가 보입니다 — 복사본이 아니라 **별칭
(alias)** 이기 때문입니다. `torch.utils.dlpack.from_dlpack`도 같은 동작을 하는 구버전
경로이며, PyTorch 1.10+ 부터는 `torch.from_dlpack`으로 통일해 쓰는 것이 권장됩니다. 이 별칭
관계가 왜 중요한지(누가 메모리를 언제까지 살려둬야 하는지)는 6절에서 자세히 다룹니다.

In [ ]:
x = cp.arange(1_000_000, dtype=cp.float32)
if HAS_TORCH:
    t = torch.from_dlpack(x)              # CuPy -> torch (zero-copy)
    print('torch device:', t.device, '| 같은 포인터?', t.data_ptr() == x.data.ptr)
    t += 1                                # torch에서 수정하면
    print('CuPy에도 반영?', float(x[0]))  # 공유 메모리라 반영됨
else:
    print('torch 없음 — 개념: torch.from_dlpack(cupy_array)')

<a id="4"></a>
## 4. PyTorch → CuPy (무복사)

반대로 `cp.from_dlpack(torch_tensor)` 로 PyTorch 텐서를 CuPy 배열로 봅니다.

### 조금 더 구체적으로

방향만 바뀔 뿐 메커니즘은 3절과 동일합니다 — CuPy가 텐서의 `__dlpack__()`을 호출해 같은
포인터를 가리키는 `cupy.ndarray`를 만듭니다. 실무에서 자주 걸리는 함정 하나는 **자동미분
(autograd) 그래프가 연결된 텐서**입니다. `requires_grad=True`인 텐서는 역전파를 위한 계산
그래프를 물고 있어서, 그 상태 그대로 DLPack으로 내보내면 오류가 나거나(버전에 따라) 그래프가
끊어질 수 있습니다. CuPy로 넘기기 전에는 보통 `tensor.detach()`로 그래프에서 분리한 뒤
넘겨야 안전합니다 — 순전파 결과만 필요하고 그 이후 CuPy에서 순수 후처리를 한다면 문제 없지만,
CuPy 쪽 연산 결과를 다시 PyTorch 학습 루프에 넣어 역전파해야 한다면 이 무복사 다리를 건널 때
그래프가 끊긴다는 점을 설계에 반영해야 합니다. `13_dl_preprocess_capstone`에서 실제 학습
루프에 CuPy 전처리를 끼워 넣을 때 이 디테일이 다시 등장합니다.

In [ ]:
if HAS_TORCH:
    t = torch.ones(1_000_000, device='cuda', dtype=torch.float32)
    g = cp.from_dlpack(t)                 # torch -> CuPy (zero-copy)
    print('같은 포인터?', g.data.ptr == t.data_ptr())
    g *= 2                                 # CuPy에서 수정
    print('torch에도 반영?', float(t[0]))
else:
    print('torch 없음 — 개념: cp.from_dlpack(torch_tensor)')

<a id="5"></a>
## 5. `__cuda_array_interface__`

CuPy 배열은 `__cuda_array_interface__`(CAI) 속성으로 **GPU 포인터·shape·dtype·strides**를 노출합니다.
Numba·PyTorch 등 CAI를 이해하는 라이브러리는 이를 보고 무복사로 접근합니다.

### 조금 더 구체적으로: DLPack과 무엇이 다른가

CAI는 DLPack보다 훨씬 가벼운 규약입니다. 별도의 C 구조체나 캡슐 없이, 그냥 **파이썬 객체의
속성(attribute)** 하나(`__cuda_array_interface__`)가 아래와 같은 평범한 딕셔너리를 반환하면
끝입니다.

| 키 | 의미 |
|----|------|
| `shape` | 배열 모양 |
| `typestr` | dtype 문자열(예: `'<f4'` = little-endian float32) |
| `data` | `(포인터 정수, 읽기전용 여부)` 튜플 |
| `strides` | 보폭(연속 배열이면 `None`) |
| `version` | 규약 버전 |
| `stream` | (v3+) 이 데이터가 준비된 시점을 나타내는 스트림 동기화 정보 |

CAI는 원래 **Numba 프로젝트**가 GPU 배열을 라이브러리 종류와 무관하게 커널에 넘기기 위해
제안한 규약으로, 이후 CuPy·RAPIDS(cuDF)·(구버전) PyTorch 등이 채택했습니다. DLPack이 C ABI
수준의 "정식 표준"이라면, CAI는 **"파이썬 덕타이핑으로 충분히 실용적인 규약"** 에 가깝습니다 —
구현이 쉬운 대신 순수 파이썬 세계 밖(C++/다른 언어 런타임)으로는 그대로 들고 나가기 어렵습니다.
실무에서는 "Numba 커널에 CuPy 배열을 바로 넣고 싶다 → CAI", "PyTorch·TensorFlow처럼 무거운
프레임워크와 정식으로 텐서를 주고받고 싶다 → DLPack" 정도로 구분해 생각하면 됩니다. 8절에서
Numba가 CAI를 어떻게 활용하는지 실제 커널로 확인합니다.

In [ ]:
x = cp.arange(6, dtype=cp.float32).reshape(2,3)
cai = x.__cuda_array_interface__
print('shape:', cai['shape'], '| typestr:', cai['typestr'])
print('data ptr:', cai['data'][0] == x.data.ptr)   # 디바이스 포인터

<a id="6"></a>
## 6. 주의점

무복사 공유의 본질은 "두 개의 파이썬 객체가 같은 물리 메모리를 가리킨다"는 것입니다. 이는
성능상 이득이지만, 동시에 **"누가 이 메모리를 언제까지 책임지는가"** 라는 질문을 반드시 동반합니다.
아래 네 가지는 실전에서 가장 자주 사고로 이어지는 지점입니다.

- **소유권/수명**: 원본 배열이 살아 있어야 공유 뷰가 유효(원본이 해제되면 위험)
- **동기화/스트림**: 서로 다른 스트림에서 같은 메모리를 쓰면 이벤트로 순서 보장 필요(06)
- **dtype/연속성**: 일부 변환은 연속(C-contiguous)·지원 dtype을 요구
- 무복사는 **같은 디바이스**에서만 — 다른 GPU면 전송 필요

### 조금 더 구체적으로

**1) 소유권/수명(ownership)** — DLPack/CAI로 만든 뷰 객체는 대부분 원본 객체에 대한 참조를
내부적으로 붙들고 있어(예: DLPack의 `deleter` 콜백, 혹은 파이썬 참조 카운트), 뷰가 살아있는
동안에는 원본이 가비지 컬렉션되지 않도록 설계되어 있습니다. 하지만 이는 **"파이썬 참조 카운트가
연결되어 있을 때"** 의 이야기이고, `cupy.cuda.MemoryPool`(05절)처럼 메모리를 풀링·재사용하는
계층까지 포함하면 이야기가 더 미묘해집니다 — 명시적으로 `del`을 호출하거나 스코프를 벗어나
CuPy 배열이 예상보다 일찍 회수되면, 다른 프레임워크가 들고 있는 뷰가 **이미 재사용된 메모리**를
가리키는 use-after-free 상태가 될 수 있습니다. 원칙은 단순합니다: **무복사 뷰를 쓰는 동안에는
원본 변수에 대한 참조를 명시적으로 살려두세요.**

**2) 동기화/스트림** — 가장 반직관적인 함정입니다. CuPy와 PyTorch는 (기본적으로는 같은 디바이스의
기본 스트림을 쓰지만) 각자 스트림을 만들어 쓸 수 있고, **서로 다른 스트림의 작업은 GPU가 순서
보장 없이 동시에 실행**합니다. 예를 들어 CuPy가 스트림 A에서 배열 `z`를 계산하는 도중에, 아직
연산이 끝나지 않은 `z`를 스트림 B에서 PyTorch가 `from_dlpack`으로 읽어버리면 **아직 쓰이지 않은
쓰레기 값**을 읽는 레이스 컨디션이 발생할 수 있습니다. DLPack v0.8부터는 `__dlpack__(stream=...)`
인자로 "이 스트림에서 쓸 것이다"를 알려 프레임워크가 자동으로 동기화하도록 하는 매커니즘이
추가되었지만, 모든 프레임워크·버전이 이를 완전히 구현하지는 않습니다. 안전하게 가려면 무복사
핸드오프 전후로 `cp.cuda.get_current_stream().synchronize()`(또는 `torch.cuda.synchronize()`)로
명시적으로 동기화하거나, 06절(`06_streams_async`)에서 배우는 **CUDA 이벤트**로 "이 이벤트가
끝날 때까지 기다려"라는 순서 보장을 걸어야 합니다. 8절 말미에서 Numba 커널과 CuPy 사이의
구체적인 레이스 시나리오와 이벤트 동기화 해법을 다룹니다.

**3) dtype/연속성** — 무복사 변환은 대상 프레임워크가 이해할 수 있는 dtype(대부분 표준 수치형)과,
많은 경우 **C-contiguous** 메모리 레이아웃을 전제로 합니다. 전치(`.T`)나 부분 슬라이싱으로 만든
비연속 배열을 그대로 넘기면 strides 해석이 어긋나 잘못된 값을 읽거나, 아예 변환이 거부될 수
있습니다. 이 문제와 해결책(`cp.ascontiguousarray()`)은 8절에서 실제 사례로 더 깊이 다룹니다.

**4) 디바이스 범위** — DLPack/CAI의 무복사는 물리적으로 **같은 GPU** 안에서만 성립합니다.
포인터 자체가 특정 디바이스의 주소 공간을 가리키기 때문에, 텐서를 다른 GPU(멀티 GPU 환경)로
옮기려면 결국 NVLink나 PCIe를 통한 **실제 전송**(P2P 복사 또는 host 경유)이 필요합니다 — 이때는
"무복사"라는 이름이 무색해지므로, 멀티 GPU 파이프라인을 설계할 때는 어느 단계에서 디바이스가
바뀌는지 미리 파악해두는 것이 중요합니다.

<a id="7"></a>
## 7. 연습 — CuPy 전처리 → PyTorch → CuPy (무복사)

CuPy로 표준화한 뒤 **무복사로 PyTorch에 넘겨** 연산하고, 다시 CuPy로 받아 후처리하는 함수를 완성하세요(전송 0).

이 연습은 실전 파이프라인의 축소판입니다: **전처리(CuPy) → 연산(PyTorch) → 후처리(CuPy)** 세 단계를
한 함수 안에서 연결하되, 각 단계 전환마다 `from_dlpack()`만 사용하고 `asnumpy`/`asarray`는 단 한
번도 등장하지 않아야 합니다. `z`가 함수 안에서 참조되는 동안에는 6절의 "소유권" 규칙에 따라 안전하게
살아있으므로, 이 규모의 짧은 함수에서는 별도의 동기화 코드 없이도 무복사 왕복이 안전합니다(같은
파이썬 스코프·같은 기본 스트림이기 때문). `13_dl_preprocess_capstone`에서는 이 패턴이 실제 학습
루프의 배치 단위 전처리로 확장됩니다.

In [ ]:
def pipeline(x_cp):
    # 1) CuPy 전처리: 표준화
    z = (x_cp - x_cp.mean()) / (x_cp.std() + 1e-6)
    if not HAS_TORCH:
        return z
    # TODO: t = torch.from_dlpack(z); t = torch.relu(t)   # 무복사 + torch 연산
    # TODO: return cp.from_dlpack(t)                       # 다시 CuPy로
    raise NotImplementedError

x = cp.random.randn(1_000_000, dtype=cp.float32)
# out = pipeline(x); print(type(out))

전송이 정말 0인지 확인하려면, `pipeline` 실행 전후로 `nvidia-smi`의 PCIe 처리량 카운터나
`cp.cuda.runtime.getDeviceProperties`가 아니라 — 더 간단히는 이 함수 안 어디에도 `asnumpy`/
`asarray`가 없다는 것 자체가 증거입니다. 해답에서 `torch` 미설치 시 `cp.maximum(z, 0)`으로
동일한 수학적 결과(ReLU)를 CuPy만으로 재현해 두어, 환경에 관계없이 같은 로직을 검증할 수 있게
했습니다.

<details><summary>💡 해답 보기</summary>

```python
def pipeline(x_cp):
    z = (x_cp - x_cp.mean()) / (x_cp.std() + 1e-6)
    if not HAS_TORCH:
        return cp.maximum(z, 0)
    t = torch.from_dlpack(z)      # zero-copy
    t = torch.relu(t)
    return cp.from_dlpack(t)      # zero-copy back

x = cp.random.randn(1_000_000, dtype=cp.float32)
out = pipeline(x)
# 검증: relu(표준화) 와 일치
z = (x - x.mean())/(x.std()+1e-6)
allclose(cp.maximum(z,0), out, rtol=1e-4, atol=1e-4, name='interop pipeline')
```
</details>

<a id="8"></a>
## 8. Numba ↔ CuPy & 추가 주제

**Numba CUDA 커널은 CuPy 배열을 직접 받습니다** — CuPy가 `__cuda_array_interface__`(CAI)를 노출하므로 무복사로 접근합니다.
즉 08~09에서 만든 Numba 커널을 CuPy 배열에 바로 적용할 수 있습니다.

### 조금 더 구체적으로

5절에서 본 것처럼 CAI는 원래 Numba가 제안한 규약이라, `numba.cuda`가 CAI를 이해하는 것은
당연한 결과에 가깝습니다 — CuPy 배열을 커널 인자로 그대로 넘기면 Numba는 `__cuda_array_interface__`
딕셔너리에서 포인터·shape·dtype·strides를 읽어 **자신이 직접 GPU 메모리에 접근**합니다. 여기서
중요한 점은 이 접근에 **어떤 안전장치도 없다**는 것입니다 — CAI는 "지금 이 포인터를 읽어도
안전한지"(즉 그 위에서 실행 중인 이전 연산이 끝났는지)를 검증해주지 않습니다. `08_numba_basics`/
`09`에서 작성한 커널을 CuPy 배열에 바로 적용할 때, 두 라이브러리의 연산이 **서로 다른 스트림**에
걸려 있다면 6절에서 설명한 레이스 컨디션이 그대로 재현될 수 있습니다. 아래 코드 셀은 가장 단순한
형태(같은 기본 스트림, 동기 실행)라 안전하지만, 바로 다음 마크다운 셀에서 이 문제를 실제 시나리오로
더 파고듭니다.

In [ ]:
try:
    from numba import cuda
    @cuda.jit
    def add_one(a):
        i = cuda.grid(1)
        if i < a.size: a[i] += 1.0
    g = cp.arange(16, dtype=cp.float32)
    add_one[1, 16](g)            # Numba가 CuPy 배열을 CAI로 직접 수정(무복사·in-place)
    print(cp.asnumpy(g))         # 1..16
except Exception as e:
    print('numba 미설치:', e)

6절과 8절에서 언급한 두 가지 주의점 — **연속성(contiguity)** 과 **스트림 동기화** — 을
조금 더 구체적인 메커니즘과 API 관점에서 풀어봅니다.

**2D·dtype 주의**: 무복사 공유는 보통 **연속(C-contiguous)·지원 dtype**을 요구합니다. 
GPU 메모리는 본질적으로 1차원의 긴 선형 구조입니다. 2D 이상의 다차원 배열을 이 1차원 메모리에 어떻게 구겨 넣느냐에 따라 연속성(Contiguity)이 결정됩니다.
- C-Contiguous (행 기준 연속)란?
  * 기본적으로 NumPy나 CuPy에서 2D 배열을 생성하면 C-Contiguous (C-order) 방식으로 메모리에 저장됩니다. 이는 같은 행(Row)에 있는 데이터가 메모리상에 나란히(연속적으로) 배치된다는 뜻입니다.

- 비연속 데이터가 발생하는 경우
    * 배열을 생성할 때는 연속적이지만, 배열을 조작하다 보면 메모리는 그대로인데 뷰(View)만 바뀌면서 비연속 데이터가 됩니다.
    * 전치 (Transpose): arr.T를 호출하면 행과 열을 읽는 순서만 바뀔 뿐 메모리는 재배치되지 않습니다. (C-order가 F-order로 변환됨)
    * 슬라이싱 (Slicing): arr[:, 1:3]처럼 특정 열만 추출하면, 메모리상에서는 듬성듬성 떨어진 데이터를 건너뛰며(Stride) 읽어야 합니다.

- 왜 Numba에서 문제가 될까?
    * CuPy 배열이 비연속 상태일 때 이를 Numba 커널에 넘기면 두 가지 문제가 발생할 수 있습니다.
       * 데이터 오염 및 충돌: Numba 커널이 데이터의 보폭(Stride) 정보를 완벽하게 해석하지 못하거나 무시하고 선형적으로 메모리에 접근하면, 전혀 엉뚱한 값을 읽거나 쓰게 됩니다.
       * 성능 저하 (Memory Coalescing 실패): GPU는 여러 스레드가 한 번에 인접한 메모리를 덩어리째 읽어오는(Coalesced Memory Access) 방식으로 속도를 냅니다. 데이터가 듬성듬성 떨어져 있으면 메모리 접근 횟수가 급증하여 성능이 폭락합니다.
    * 해결책: cp.ascontiguousarray()
       * 비연속 배열을 Numba 커널에 넘기기 전에는 반드시 메모리를 연속된 공간에 새로 복사하여 재배열해야 합니다.

- 스트림(Stream)과 이벤트(Event) 동기화
    * GPU는 여러 작업을 동시에 처리할 수 있는 비동기(Asynchronous) 장치입니다. 
    * 스트림(Stream)은 GPU에 내리는 명령(데이터 복사, 커널 실행 등)의 대기열(Queue)입니다.
    * 스트림 분리로 인한 레이스 컨디션(Race Condition)
       * 같은 스트림에 들어간 명령은 순서대로 실행되지만, 서로 다른 스트림에 들어간 명령은 GPU가 동시에 병렬로 실행합니다.
       * CuPy 작업과 Numba CUDA 커널이 서로 다른 스트림을 사용하도록 설정되어 있다면 심각한 문제가 발생할 수 있습니다.
           * 상황: CuPy로 A 배열에 연산을 수행한 뒤, 그 결과를 Numba 커널이 받아서 B 배열에 쓴다고 가정해 봅시다.
           * 문제: CuPy가 A 배열에 값을 다 쓰기도 전에, Numba 커널이 동시에 실행되어 아직 계산되지 않은 쓰레기 값을 읽어갈 수 있습니다.
    * 이벤트(Event)를 이용한 순서 보장
       * 이러한 충돌을 막기 위해 두 스트림 간에 "이 작업이 끝날 때까지 기다려"라는 신호등을 세워야 합니다. 이때 사용하는 것이 CUDA 이벤트(Event)입니다.
           * CuPy 연산이 끝나는 지점에 이벤트(Event)를 기록(Record)합니다.
           * Numba 커널이 실행될 스트림에게 해당 이벤트가 완료될 때까지 대기(Wait)하라고 지시합니다.



이 두 가지(연속성 보장, 이벤트 기반 순서 보장)는 단순한 배경지식이 아니라 실전 체크리스트입니다.
`13_dl_preprocess_capstone`에서 커스텀 CUDA 커널로 만든 전처리 결과를 PyTorch 학습 루프에
연결할 때, "이 배열이 연속인가?"와 "이 스트림 작업이 다 끝났는가?"라는 두 질문을 매번 다시
떠올리게 될 것입니다.

**연습 — 무복사 왕복 & 소유권 확인**: CuPy 배열을 (있으면)torch로 무복사 변환→torch에서 수정→**원본 CuPy에도 반영**되는지 확인하세요.

이 연습은 방향을 반대로 뒤집어(PyTorch → CuPy) 6절의 "소유권" 논의를 몸으로 확인하는 실습입니다.
`torch.from_dlpack(x)`로 만든 텐서 `t`는 새 메모리가 아니라 `x`와 **같은 물리 주소**를 가리키므로,
`t.add_(5)`처럼 PyTorch 쪽에서 in-place로 수정하면 원본 CuPy 배열 `x`를 다시 읽었을 때도 그
변화가 그대로 보여야 합니다. 만약 두 값이 다르게 나온다면 어딘가에서 복사가 일어났다는 뜻이므로,
"정말 무복사인가"를 스스로 검증하는 습관을 들이는 셀입니다.

In [ ]:
x = cp.zeros(8, dtype=cp.float32)
# TODO: HAS_TORCH면 t=torch.from_dlpack(x); t.add_(5); print(cp.asnumpy(x))  # 5로 채워졌나?
# 같은 메모리를 공유하므로 torch 수정이 CuPy에 보여야 함

포인터 동일성(`t.data_ptr() == x.data.ptr`)까지 함께 확인하면, "값이 우연히 같다"가
아니라 "애초에 같은 메모리를 보고 있다"는 것을 직접 증명할 수 있습니다. `torch` 미설치 환경에서는
개념 설명으로 대체하여, 실습 환경에 관계없이 노트북이 끝까지 실행되도록 했습니다.

<details><summary>💡 해답 보기</summary>

```python
if HAS_TORCH:
    t = torch.from_dlpack(x)   # zero-copy view
    t.add_(5)                  # torch에서 in-place 수정
    print(cp.asnumpy(x))       # [5,5,...] — 공유 메모리라 반영
    print('포인터 동일?', t.data_ptr() == x.data.ptr)
else:
    print('torch 없음 — 개념만: from_dlpack은 메모리를 공유한다')
```
</details>

### 체크포인트
- [ ] DLPack `from_dlpack`로 무복사 교환을 이해했다
- [ ] CuPy↔PyTorch를 포인터 공유로 변환했다
- [ ] `__cuda_array_interface__`의 역할을 안다
- [ ] 소유권·동기화 주의점을 안다
- [ ] DLPack과 `__cuda_array_interface__`의 차이(ABI 표준 vs 파이썬 속성 규약)를 설명할 수 있다
- [ ] 무복사 공유에서 소유권·스트림 동기화 문제가 왜, 어떻게 생기는지 시나리오로 설명할 수 있다

다음: **`13_dl_preprocess_capstone`** — Day 2 종합 캡스톤(커스텀 커널 + interop). 여기서 배운 무복사 연동을 커스텀 CUDA 커널 전처리와 PyTorch 학습 루프에 실제로 이어붙입니다.